**Imports**

In [0]:
import pyspark.sql.functions as F

**Tables**

In [0]:
rental_df=spark.read.table("rental.bike_rental_bronze.taken_rent")

In [0]:
silver_bike_df=spark.read.table("rental.bike_rental_silver.bike")

In [0]:
silver_customer_df=spark.read.table("rental.bike_rental_silver.customer")

**Joining Rental with Customer and Bike**

In [0]:
# Merging rental with customer and bike tables in the silver layer enriches the rental data with customer and bike details.
# Retain and display columns: rental id, customer_id, customer_name, bike_id, bike_type, duration, start_timestamp, total_paid.

silver_rental_df = (
    rental_df
    .join(silver_bike_df, rental_df.bike_id == silver_bike_df.id, "inner")
    .join(silver_customer_df, rental_df.customer_id == silver_customer_df.id, "inner")
    .select(
        rental_df.id.alias("rental_id"),
        rental_df.customer_id,
        silver_customer_df.name.alias("customer_name"),
        rental_df.bike_id,
        silver_bike_df.model.alias("bike_type"),
        rental_df.duration,
        rental_df.start_timestamp,
        rental_df.total_paid
    )
)

**End Timestamp**

In [0]:
silver_rental_df = silver_rental_df.withColumn(
    "end_timestamp",
    F.from_unixtime(
        F.unix_timestamp("start_timestamp") + F.col("duration") * 60
    ).cast("timestamp")
)

**Rental Type**

In [0]:
silver_rental_df = silver_rental_df.withColumn(
    "rental_type",
    F.when(F.col("duration") <= 1440, "hourly")
    .when(F.col("duration") > 1440, "daily")
    .otherwise("other")
)

**Rental_Revenue**

In [0]:
silver_rental_df = silver_rental_df.withColumn(
    "rental_revenue", F.col("total_paid") / (F.col("duration") / 60))

**Rental Based On Different Parts of Time**

In [0]:
silver_rental_df = (
    silver_rental_df.withColumn("rental_day", F.dayofmonth("start_timestamp"))
    .withColumn("rental_month", F.month("start_timestamp"))
    .withColumn("rental_year", F.year("start_timestamp")).withColumn("rental_weekday", F.date_format("start_timestamp", "EEEE")).withColumn("rental_hour",F.hour("start_timestamp"))
)

**Duration Bucket**

In [0]:
silver_rental_df = silver_rental_df.withColumn(
    "duration_bucket",
    F.when(F.col("duration") <= 240, "short")
     .when(F.col("duration") <= 720, "medium")
     .otherwise("long")
)

In [0]:
silver_rental_df.write.mode("overwrite").saveAsTable("rental.bike_rental_silver.rental")